In [1]:
from google.colab import files
import gzip, json, os, re, math
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
uploaded = files.upload()
json_gz_files = [name for name in uploaded if name.endswith('.json.gz')]
if not json_gz_files:
    raise FileNotFoundError('')
DATA_PATH = json_gz_files[0]
print('Dataset:', DATA_PATH)
def load_jsonl_gz(path):
    doc_ids, documents, timestamps, urls = [], [], [], []
    with gzip.open(path, 'rt', encoding='utf-8') as handle:
        for line_no, line in enumerate(handle):
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'Line {line_no} is not a JSON object.')
            text = str(record.get('text', '') or '').strip()
            if not text:
                continue
            doc_ids.append(f'D{len(doc_ids):05d}')
            documents.append(text)
            timestamps.append(str(record.get('timestamp', '') or ''))
            urls.append(str(record.get('url', '') or ''))
    return doc_ids, documents, timestamps, urls

doc_ids, documents, timestamps, urls = load_jsonl_gz(DATA_PATH)
print('Number of documents:', len(documents))
print('First document ID:', doc_ids[0])
print('First URL:', urls[0])
print('First document preview:', documents[0][:500].replace('\n',' '))

Saving c4-train.00000-of-01024-30K.json.gz to c4-train.00000-of-01024-30K.json.gz
Dataset: c4-train.00000-of-01024-30K.json.gz
Number of documents: 30000
First document ID: D00000
First URL: https://klyq.com/beginners-bbq-class-taking-place-in-missoula/
First document preview: Beginners BBQ Class Taking Place in Missoula! Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills. He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat select


# D


In [2]:
vectorizer = CountVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b')
count_matrix = vectorizer.fit_transform(documents)
N, V = count_matrix.shape
document_frequency = np.asarray((count_matrix > 0).sum(axis=0)).ravel()
idf = np.log(N / document_frequency)
row_sums = np.asarray(count_matrix.sum(axis=1)).ravel()
inv_row_sums = np.zeros_like(row_sums, dtype=float)
mask = row_sums > 0
inv_row_sums[mask] = 1.0 / row_sums[mask]
tf_matrix = sparse.diags(inv_row_sums).dot(count_matrix)
tfidf_matrix = tf_matrix.multiply(idf).tocsr()
nnz = tfidf_matrix.nnz
sparsity = 1.0 - nnz / (N * V)
print('Number of documents =', N)
print('Vocabulary size =', V)
print('Matrix shape =', tfidf_matrix.shape)
print('NNZ =', nnz)
print('Sparsity =', sparsity)
print('Sparsity (%) =', 100*sparsity)

Number of documents = 30000
Vocabulary size = 193837
Matrix shape = (30000, 193837)
NNZ = 5104560
Sparsity = 0.9991221902939067
Sparsity (%) = 99.91221902939067


In [3]:
terms = vectorizer.get_feature_names_out()
vocab_table = pd.DataFrame({'term': terms, 'df': document_frequency, 'idf': idf})
print('20 terms with highest document frequency')
display(vocab_table.sort_values(['df','term'], ascending=[False,True]).head(20))
print('20 terms with highest IDF')
display(vocab_table.sort_values(['idf','term'], ascending=[False,True]).head(20))
SELECTED_DOC_INDEX = 0
row = tfidf_matrix.getrow(SELECTED_DOC_INDEX)
selected_tfidf = pd.DataFrame({
    'term': terms[row.indices],
    'tfidf': row.data,
}).sort_values(['tfidf','term'], ascending=[False,True])
print('20 terms with highest TF-IDF in selected document:', doc_ids[SELECTED_DOC_INDEX])
display(selected_tfidf.head(20))

20 terms with highest document frequency


,term,df,idf
168465,the,27893,0.072822
17391,and,27423,0.089815
170478,to,26689,0.116946
122789,of,26031,0.141909
11464,a,25905,0.146761
86528,in,25224,0.173401
68585,for,23651,0.237792
89932,is,22739,0.277116
184990,with,21405,0.337573
123639,on,20262,0.392450


20 terms with highest IDF


,term,df,idf
4,00000,1,10.308953
5,000000,1,10.308953
6,00000000,1,10.308953
7,0000000000000965,1,10.308953
8,00000001,1,10.308953
9,00000048,1,10.308953
10,000002,1,10.308953
11,000004b926f1,1,10.308953
12,00000781,1,10.308953
13,0000085054,1,10.308953


20 terms with highest TF-IDF in selected document: D00000


,term,tfidf
7,bbq,0.178414
14,class,0.094085
6,balay,0.078694
39,kcbs,0.073403
44,meat,0.072493
42,lonestar,0.070308
45,missoula,0.061117
4,apron,0.056630
66,smoker,0.056218
79,timelines,0.052481


1,
Không. Một term xuất hiện trong rất nhiều documents có DF cao, nên IDF của term đó thấp. Vì TF-IDF được tính bằng `TF × IDF`, một term rất phổ biến thường không có TF-IDF cao nếu nó không có tần suất đặc biệt cao trong document đang xét. Kết quả thực nghiệm cũng cho thấy các term phổ biến như `the`, `and`, `to`, `of` có DF rất lớn nhưng IDF lần lượt chỉ khoảng 0.073, 0.090, 0.117 và 0.142.

2,
Không. IDF cao chỉ cho biết term đó hiếm trong toàn corpus. Để có TF-IDF cao, term đó còn phải xuất hiện với TF đủ lớn trong document đang xét. Nếu một term có IDF cao nhưng không xuất hiện trong một document thì TF bằng 0 và TF-IDF cũng bằng 0. Trong kết quả, nhiều token chỉ xuất hiện một lần có IDF rất cao, khoảng 10.309, nhưng điều đó không có nghĩa chúng có TF-IDF cao trong mọi document.

3,
Vì tất cả documents phải được biểu diễn trên cùng một vocabulary gồm V terms để có thể so sánh với nhau bằng cosine similarity. Một document chỉ chứa một số ít terms nên phần lớn các phần tử trong vector bằng 0, tạo thành sparse vector. Tuy nhiên, vector vẫn phải có đủ V chiều để vị trí của mỗi term có ý nghĩa thống nhất giữa tất cả documents.


# F

In [4]:
from sklearn.model_selection import train_test_split
def tokenize_a(text):
    return re.findall(r'\b\w+\b', text.lower(), flags=re.UNICODE)

def normalize_punctuation(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_b(text):
    return [t for t in normalize_punctuation(text).split() if t not in ENGLISH_STOP_WORDS]
train_docs, test_docs = train_test_split(documents, test_size=0.2, random_state=42)
train_a_vocab = {t for d in train_docs for t in tokenize_a(d)}
train_b_vocab = {t for d in train_docs for t in tokenize_b(d)}
def token_oov_rate(test_docs, vocab, tokenizer):
    total = oov = 0
    for doc in test_docs:
        toks = tokenizer(doc)
        total += len(toks)
        oov += sum(t not in vocab for t in toks)
    return 0.0 if total == 0 else oov / total
a_tokens = [tokenize_a(d) for d in documents]
b_tokens = [tokenize_b(d) for d in documents]
a_vocab = set(t for d in a_tokens for t in d)
b_vocab = set(t for d in b_tokens for t in d)
vectorizer_a = CountVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b')
matrix_a = vectorizer_a.fit_transform(documents)
vectorizer_b = CountVectorizer(preprocessor=normalize_punctuation, tokenizer=str.split, token_pattern=None, stop_words='english')
matrix_b = vectorizer_b.fit_transform(documents)
!pip -q install transformers
from transformers import AutoTokenizer
TOKENIZER_NAME = 'bert-base-uncased'
subword_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
c_tokens = [subword_tokenizer.tokenize(d, add_special_tokens=False) for d in documents]
c_vocab = set(t for d in c_tokens for t in d)
c_texts = [' '.join(toks) for toks in c_tokens]
c_vectorizer = CountVectorizer(lowercase=False, tokenizer=str.split, token_pattern=None)
matrix_c = c_vectorizer.fit_transform(c_texts)
def matrix_sparsity(m):
    return 1.0 - m.nnz / (m.shape[0] * m.shape[1])
rows = [
    {'pipeline':'A','vocabulary_size':len(a_vocab),'average_tokens_per_document':np.mean([len(x) for x in a_tokens]),'matrix_sparsity':matrix_sparsity(matrix_a),'oov_or_unk_rate':token_oov_rate(test_docs, train_a_vocab, tokenize_a)},
    {'pipeline':'B','vocabulary_size':len(b_vocab),'average_tokens_per_document':np.mean([len(x) for x in b_tokens]),'matrix_sparsity':matrix_sparsity(matrix_b),'oov_or_unk_rate':token_oov_rate(test_docs, train_b_vocab, tokenize_b)},
]
unk_total = sum(sum(t == subword_tokenizer.unk_token for t in toks) for toks in [subword_tokenizer.tokenize(d, add_special_tokens=False) for d in test_docs])
sub_total = sum(len(subword_tokenizer.tokenize(d, add_special_tokens=False)) for d in test_docs)
rows.append({'pipeline':'C','vocabulary_size':len(c_vocab),'average_tokens_per_document':np.mean([len(x) for x in c_tokens]),'matrix_sparsity':matrix_sparsity(matrix_c),'oov_or_unk_rate':0.0 if sub_total==0 else unk_total/sub_total})
preprocessing_stats = pd.DataFrame(rows)
display(preprocessing_stats)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


,pipeline,vocabulary_size,average_tokens_per_document,matrix_sparsity,oov_or_unk_rate
0,A,193837,369.696367,0.999122,0.019400
1,B,193521,197.413233,0.999354,0.035979
2,C,28339,465.099200,0.993200,0.000296


In [5]:
def build_tfidf_index_from_counts(count_matrix):
    N_local, V_local = count_matrix.shape
    df_local = np.asarray((count_matrix > 0).sum(axis=0)).ravel()
    idf_local = np.log(N_local / df_local)
    row_sums_local = np.asarray(count_matrix.sum(axis=1)).ravel()
    inv = np.zeros_like(row_sums_local, dtype=float)
    mask = row_sums_local > 0
    inv[mask] = 1.0 / row_sums_local[mask]
    tf = sparse.diags(inv).dot(count_matrix)
    return tf.multiply(idf_local).tocsr(), idf_local

def search_sparse(query, vectorizer, tfidf_matrix_local, idf_local, top_k=5):
    query_counts = vectorizer.transform([query])
    q_len = float(query_counts.sum())
    if q_len == 0:
        return []
    q_tf = query_counts.multiply(1.0/q_len)
    q_tfidf = q_tf.multiply(idf_local).tocsr()
    dots = (q_tfidf @ tfidf_matrix_local.T).toarray().ravel()
    q_norm = float(np.sqrt(q_tfidf.multiply(q_tfidf).sum()))
    d_norms = np.sqrt(tfidf_matrix_local.multiply(tfidf_matrix_local).sum(axis=1)).A1
    den = q_norm * d_norms
    sims = np.divide(dots, den, out=np.zeros_like(dots), where=den != 0)
    idx = np.argsort(-sims, kind='stable')[:top_k]
    return [(int(i), float(sims[i])) for i in idx]
tfidf_a, idf_a = build_tfidf_index_from_counts(matrix_a)
tfidf_b, idf_b = build_tfidf_index_from_counts(matrix_b)
tfidf_c, idf_c = build_tfidf_index_from_counts(matrix_c)
def b_query_transform(q): return ' '.join(tokenize_b(q))
def c_query_transform(q): return ' '.join(subword_tokenizer.tokenize(q, add_special_tokens=False))
queries = ['medical image classification','transformer language model','deep learning healthcare','natural language processing']
search_comparison = []
for q in queries:
    for name, q2, vec, mat, idfv in [
        ('A',q,vectorizer_a,tfidf_a,idf_a),
        ('B',b_query_transform(q),vectorizer_b,tfidf_b,idf_b),
        ('C',c_query_transform(q),c_vectorizer,tfidf_c,idf_c),
    ]:
        r = search_sparse(q2, vec, mat, idfv, top_k=5)
        search_comparison.append({'query':q,'pipeline':name,'top5':[doc_ids[i] for i,_ in r],'scores':[round(s,6) for _,s in r]})
display(pd.DataFrame(search_comparison))

,query,pipeline,top5,scores
0,medical image classification,A,"[D18971, D08527, D19908, D15682, D08370]","[0.444091, 0.367154, 0.233654, 0.226001, 0.220..."
1,medical image classification,B,"[D18971, D08527, D19908, D15682, D12658]","[0.446573, 0.367839, 0.235495, 0.231945, 0.228..."
2,medical image classification,C,"[D18971, D08527, D19908, D15682, D08370]","[0.471326, 0.367946, 0.234644, 0.228185, 0.217..."
3,transformer language model,A,"[D27936, D25428, D24482, D00701, D04289]","[0.500487, 0.267496, 0.205812, 0.201602, 0.199..."
4,transformer language model,B,"[D27936, D25428, D24482, D00701, D04289]","[0.505581, 0.278216, 0.219173, 0.204312, 0.200..."
5,transformer language model,C,"[D27936, D25428, D24482, D00701, D04075]","[0.382788, 0.325078, 0.260205, 0.253583, 0.25205]"
6,deep learning healthcare,A,"[D11119, D11979, D06123, D09252, D11777]","[0.307795, 0.307256, 0.305914, 0.297339, 0.247..."
7,deep learning healthcare,B,"[D11119, D06123, D11979, D09252, D07564]","[0.312871, 0.312067, 0.310618, 0.299105, 0.281..."
8,deep learning healthcare,C,"[D11979, D09252, D06123, D11119, D11777]","[0.31055, 0.308695, 0.307704, 0.296013, 0.249518]"
9,natural language processing,A,"[D25428, D08705, D24482, D00701, D04075]","[0.374109, 0.369019, 0.287841, 0.281953, 0.276..."


1,
Lowercasing làm các token viết hoa và viết thường được đưa về cùng một dạng, vì vậy các biến thể khác nhau về chữ hoa/chữ thường được gộp lại thành một term. Điều này giúp giảm số lượng vocabulary và làm representation nhất quán hơn. Tuy nhiên, từ kết quả search, việc thay đổi vocabulary không làm ranking của tất cả các query thay đổi đáng kể.
2,
Không. Stopword removal có thể làm vocabulary nhỏ hơn và giảm các từ ít mang tính phân biệt, nhưng không đảm bảo search luôn tốt hơn. Trong bảng kết quả, Pipeline B có ranking rất gần Pipeline A ở nhiều query và chỉ thay đổi một số document ở vị trí cuối Top-5. Vì vậy, stopword removal nên được đánh giá dựa trên nhiệm vụ search cụ thể thay vì giả định rằng loại bỏ càng nhiều từ thì representation càng tốt.
3,
Loại punctuation có thể làm mất thông tin về cấu trúc và ngữ cảnh của text. Ví dụ, dấu câu có thể biểu diễn ranh giới câu, viết tắt, biểu thức đặc biệt, hoặc sự khác biệt giữa các chuỗi ký tự. Khi punctuation bị loại bỏ, một số token hoặc biểu thức có thể bị gộp lại hoặc mất thông tin phân biệt. Do đó, punctuation normalization có thể giúp vocabulary gọn hơn nhưng cũng có thể làm mất một phần thông tin hữu ích.
4,
Pipeline **A** là sparse nhất.
Kết quả đo được:
* Pipeline A: sparsity ≈ 99.9122%
* Pipeline B: sparsity ≈ 89.4896%
* Pipeline C: sparsity ≈ 99.3200%
Do đó, theo metric sparsity của experiment, thứ tự là: A > C > B
Pipeline A có vocabulary rất lớn nên phần lớn các vị trí trong TF-IDF matrix bằng 0.
5,
Dựa vào bảng Top-5 và scores hiện tại, chưa thể kết luận Pipeline A, B hay C có search performance tốt hơn. Các pipeline thường trả về những document gần giống nhau nhưng thứ tự có thay đổi.
Ví dụ:
* `medical image classification`: A và C có cùng Top-5; B chỉ thay D08370 bằng D12658 ở vị trí 5.
* `transformer language model`: A và B có cùng Top-5; C thay D04289 bằng D04075.
* `deep learning healthcare`: cả ba pipeline đều trả về các document rất giống nhau nhưng thứ tự thay đổi.
* `natural language processing`: A và B có cùng Top-5; C đổi vị trí D08705 và D25428.
Muốn xác định pipeline nào tốt hơn, cần tính và so sánh P@5, Recall@5 và MRR bằng relevance labels. Chỉ nhìn similarity scores không đủ để đánh giá retrieval quality.
6,
Không. Vocabulary size và search quality là hai yếu tố khác nhau. Một vocabulary nhỏ hơn có thể làm representation gọn hơn nhưng đồng thời có thể loại bỏ những terms hữu ích cho việc phân biệt documents. Ngược lại, vocabulary lớn hơn có thể giữ nhiều thông tin lexical hơn nhưng cũng tạo ra representation rất sparse và chứa nhiều noise.

# G

In [6]:
def run_search_queries(queries, top_k=5):
    rows=[]
    for q in queries:
        results = search_sparse(q, vectorizer_a, tfidf_a, idf_a, top_k=top_k)
        for rank,(idx,score) in enumerate(results,1):
            rows.append({
                'query':q,'rank':rank,'doc_id':doc_ids[idx],
                'similarity':score,'url':urls[idx],
                'preview':documents[idx][:300].replace('\n',' '),
            })
    return pd.DataFrame(rows)
search_results = run_search_queries(queries, top_k=5)
display(search_results)
search_results.to_csv('/content/results.csv', index=False)
print('Saved /content/results.csv')

,query,rank,doc_id,similarity,url,preview
0,medical image classification,1,D18971,0.444091,http://glt.rts.fi/etusivu/rts-ymparistoluokitu...,The new RTS Environmental Classification syste...
1,medical image classification,2,D08527,0.367154,https://books.google.rs/books?id=tXxQAAAAMAAJ&...,History of maize classification. How races use...
2,medical image classification,3,D19908,0.233654,https://www.wallpapersin4k.org/images/6506,Download League Of Legends Wallpapers in high-...
3,medical image classification,4,D15682,0.226001,https://itunes.apple.com/us/app/safe-text/id10...,- Group Image: Provided functionality of group...
4,medical image classification,5,D08370,0.220585,https://indiahospitaltour.com/patient-services...,What is a Online Medical Second Opinion? For o...
5,transformer language model,1,D27936,0.500487,http://www.caravan-advice.co.uk/12v-system-on-...,"hi, I am having problems with transformer / ci..."
6,transformer language model,2,D25428,0.267496,https://www.basicfacebookhelp.com/2018/11/chan...,"Note: If you're on an iPhone, you cannot chang..."
7,transformer language model,3,D24482,0.205812,https://www.eflmagazine.com/interview-harald-k...,"Harald, you are a co-owner of Language Partner..."
8,transformer language model,4,D00701,0.201602,https://www.anadolu.edu.tr/en/academics/facult...,Program in Teaching French as a Foreign Langua...
9,transformer language model,5,D04289,0.199706,http://inkakinada.com/classifieds/rent/commerc...,"Commercial Building Properties, Commercial Bui..."


Saved /content/results.csv


# H

In [7]:
manual_qrels = [
    ['medical image classification','D18971',0],
    ['medical image classification','D08527',0],
    ['medical image classification','D19908',0],
    ['medical image classification','D15682',0],
    ['medical image classification','D08370',0],
    ['transformer language model','D27936',0],
    ['transformer language model','D25428',0],
    ['transformer language model','D24482',0],
    ['transformer language model','D00701',0],
    ['transformer language model','D04289',0],
    ['deep learning healthcare','D11119',0],
    ['deep learning healthcare','D11979',0],
    ['deep learning healthcare','D06123',0],
    ['deep learning healthcare','D09252',0],
    ['deep learning healthcare','D11777',0],
    ['natural language processing','D25428',0],
    ['natural language processing','D08705',0],
    ['natural language processing','D24482',0],
    ['natural language processing','D00701',0],
    ['natural language processing','D04075',0],
]
qrels = pd.DataFrame(manual_qrels, columns=['query','doc_id','relevant'])
qrels.to_csv('/content/qrels.csv', index=False)
print('Saved /content/qrels.csv')
display(qrels)

Saved /content/qrels.csv


,query,doc_id,relevant
0,medical image classification,D18971,0
1,medical image classification,D08527,0
2,medical image classification,D19908,0
3,medical image classification,D15682,0
4,medical image classification,D08370,0
5,transformer language model,D27936,0
6,transformer language model,D25428,0
7,transformer language model,D24482,0
8,transformer language model,D00701,0
9,transformer language model,D04289,0


In [8]:
def precision_at_k(retrieved, relevant, k=5):
    relevant=set(relevant); top=retrieved[:k]
    return sum(d in relevant for d in top)/k

def recall_at_k(retrieved, relevant, k=5):
    relevant=set(relevant)
    if not relevant: return 0.0
    return sum(d in relevant for d in retrieved[:k])/len(relevant)

def reciprocal_rank(retrieved, relevant):
    relevant=set(relevant)
    for rank,d in enumerate(retrieved,1):
        if d in relevant: return 1.0/rank
    return 0.0

def evaluate_search_results(search_results, qrels_path, k=5):
    qrels=pd.read_csv(qrels_path)
    rows=[]
    for query, group in search_results.groupby('query'):
        retrieved=group.sort_values('rank')['doc_id'].tolist()
        relevant=qrels.loc[(qrels.query==query)&(qrels.relevant==1),'doc_id'].tolist()
        rows.append({'query':query,'P@5':precision_at_k(retrieved,relevant,k),'Recall@5':recall_at_k(retrieved,relevant,k),'RR':reciprocal_rank(retrieved,relevant),'#relevant':len(relevant)})
    detail=pd.DataFrame(rows)
    summary=pd.DataFrame([{'P@5':detail['P@5'].mean(),'Recall@5':detail['Recall@5'].mean(),'MRR':detail['RR'].mean()}]) if len(detail) else pd.DataFrame()
    return detail, summary

detail, summary = evaluate_search_results(search_results, '/content/qrels.csv')
display(detail)
display(summary)

if (qrels['relevant']==1).sum()==0:
    print('WARNING: current qrels contains no relevant documents, so Recall@5 and MRR are not informative.')

,query,P@5,Recall@5,RR,#relevant
0,deep learning healthcare,0.0,0.0,0.0,0
1,medical image classification,0.0,0.0,0.0,0
2,natural language processing,0.0,0.0,0.0,0
3,transformer language model,0.0,0.0,0.0,0


,P@5,Recall@5,MRR
0,0.0,0.0,0.0


# I

In [12]:
def lexical_overlap_terms(query, document):
    q=set(tokenize_a(query)); d=set(tokenize_a(document))
    return sorted(q & d)

def error_analysis_table(query):
    group = search_results[search_results['query'] == query].sort_values('rank').copy()
    group['overlap_terms'] = group['preview'].apply(
        lambda x: ', '.join(lexical_overlap_terms(query, x))
    )
    group['relevant'] = group.apply(
        lambda r: int(
            qrels.loc[
                (qrels['query'] == query) & (qrels['doc_id'] == r['doc_id']),
                'relevant'
            ].max()
        )
        if not qrels.loc[
            (qrels['query'] == query) & (qrels['doc_id'] == r['doc_id'])
        ].empty
        else np.nan,
        axis=1
    )
    return group

for q in queries:
    print('QUERY:', q)
    display(error_analysis_table(q))

QUERY: medical image classification


,query,rank,doc_id,similarity,url,preview,overlap_terms,relevant
0,medical image classification,1,D18971,0.444091,http://glt.rts.fi/etusivu/rts-ymparistoluokitu...,The new RTS Environmental Classification syste...,classification,0
1,medical image classification,2,D08527,0.367154,https://books.google.rs/books?id=tXxQAAAAMAAJ&...,History of maize classification. How races use...,classification,0
2,medical image classification,3,D19908,0.233654,https://www.wallpapersin4k.org/images/6506,Download League Of Legends Wallpapers in high-...,image,0
3,medical image classification,4,D15682,0.226001,https://itunes.apple.com/us/app/safe-text/id10...,- Group Image: Provided functionality of group...,image,0
4,medical image classification,5,D08370,0.220585,https://indiahospitaltour.com/patient-services...,What is a Online Medical Second Opinion? For o...,medical,0


QUERY: transformer language model


,query,rank,doc_id,similarity,url,preview,overlap_terms,relevant
5,transformer language model,1,D27936,0.500487,http://www.caravan-advice.co.uk/12v-system-on-...,"hi, I am having problems with transformer / ci...","model, transformer",0
6,transformer language model,2,D25428,0.267496,https://www.basicfacebookhelp.com/2018/11/chan...,"Note: If you're on an iPhone, you cannot chang...",language,0
7,transformer language model,3,D24482,0.205812,https://www.eflmagazine.com/interview-harald-k...,"Harald, you are a co-owner of Language Partner...",language,0
8,transformer language model,4,D00701,0.201602,https://www.anadolu.edu.tr/en/academics/facult...,Program in Teaching French as a Foreign Langua...,language,0
9,transformer language model,5,D04289,0.199706,http://inkakinada.com/classifieds/rent/commerc...,"Commercial Building Properties, Commercial Bui...",transformer,0


QUERY: deep learning healthcare


,query,rank,doc_id,similarity,url,preview,overlap_terms,relevant
10,deep learning healthcare,1,D11119,0.307795,https://www.harmony-alliance.eu/for-patients-c...,The opportunities offered by Big Data will onl...,healthcare,0
11,deep learning healthcare,2,D11979,0.307256,http://www.ciu.edu.tr/en/academic/institute-gr...,Doctorate of Healthcare Organization Program i...,healthcare,0
12,deep learning healthcare,3,D06123,0.305914,https://www.healthcarestudies.nz/Online-Degrees/,"With today’s advancement in technology, it is ...",,0
13,deep learning healthcare,4,D09252,0.297339,https://www.westhealth.org/press-release/west-...,"SAN DIEGO AND WASHINGTON, D.C. – Sept. 5, 2018...",healthcare,0
14,deep learning healthcare,5,D11777,0.247122,https://www.nendo.co.ke/post/african-led-visio...,It is much cheaper (50–70% cheaper) to fly to ...,,0


QUERY: natural language processing


,query,rank,doc_id,similarity,url,preview,overlap_terms,relevant
15,natural language processing,1,D25428,0.374109,https://www.basicfacebookhelp.com/2018/11/chan...,"Note: If you're on an iPhone, you cannot chang...",language,0
16,natural language processing,2,D08705,0.369019,https://foodsafetyhelpline.com/appendix-c-proc...,These regulations may be called the Food Safet...,,0
17,natural language processing,3,D24482,0.287841,https://www.eflmagazine.com/interview-harald-k...,"Harald, you are a co-owner of Language Partner...",language,0
18,natural language processing,4,D00701,0.281953,https://www.anadolu.edu.tr/en/academics/facult...,Program in Teaching French as a Foreign Langua...,language,0
19,natural language processing,5,D04075,0.276729,https://www.urbanpro.com/gurgaon/urgently-requ...,Looking for Spanish language instructor to imp...,language,0


1. Query có kết quả tốt #1: `medical image classification`
   Kết quả trả về có độ tương đồng khá cao ở tài liệu đầu tiên, nhưng phần lớn tài liệu vẫn không liên quan trực tiếp đến chủ đề cần tìm. Ví dụ, D18971 và D08527 chứa từ “classification”, còn D19908 và D15682 chứa từ “image”. Điều này cho thấy TF-IDF có thể truy hồi tài liệu chỉ vì chúng chứa một số từ trong query, dù nội dung tổng thể không phù hợp.

2. Query có kết quả tốt #2: `transformer language model`
   Tài liệu D27936 có điểm tương đồng cao nhất (0.500487) và chứa cả “transformer” và “model”. Tuy nhiên, nội dung tài liệu chủ yếu nói về transformer trong hệ thống điện chứ không phải language model. Các kết quả còn lại chủ yếu liên quan đến từ “language”. Điều này cho thấy kết quả vẫn bị chi phối bởi sự trùng khớp từ khóa riêng lẻ.

3. Query có kết quả kém #1: `deep learning healthcare`
   Các tài liệu được truy hồi chủ yếu chứa từ “healthcare” nhưng không thể hiện rõ nội dung về “deep learning”. Ví dụ, D11119 đề cập đến Big Data trong lĩnh vực y tế, trong khi D11979 và D09252 liên quan đến các chương trình và chính sách về healthcare. Điều này cho thấy TF-IDF đánh giá cao các từ trùng nhau mà chưa hiểu được mối quan hệ ngữ nghĩa giữa “deep learning” và “healthcare”.

4. Query có kết quả kém #2: `natural language processing`
   Các kết quả đầu chủ yếu liên quan đến từ “language” thay vì lĩnh vực Natural Language Processing. D25428, D24482, D00701 và D04075 đều có nội dung liên quan đến “language” nhưng không rõ ràng về NLP. D08705 lại liên quan đến “processing” trong lĩnh vực an toàn thực phẩm. Điều này cho thấy hệ thống có thể truy hồi tài liệu chỉ dựa trên từng từ riêng lẻ trong query.

5. Main failure case:
   Lỗi chính của hệ thống TF-IDF là phụ thuộc nhiều vào sự trùng khớp từ khóa và chưa hiểu được ngữ nghĩa của query. Khi query chứa các từ có nhiều nghĩa hoặc xuất hiện phổ biến như “language”, “classification”, “image” hoặc “transformer”, các tài liệu chỉ chứa một trong những từ này vẫn có thể nhận được điểm tương đồng cao dù không liên quan đến toàn bộ query. Vì vậy, TF-IDF có thể cho kết quả kém đối với những query mà ý nghĩa phụ thuộc vào sự kết hợp của nhiều từ.


# J
Có lẽ sẽ cần 1 biểu diễn cho các từ, cụm từ có liên quan về mặt ngữ nghĩa được xếp gần nhau